# 🛡️ PhishShield - Model 3: XGBoost (Tabular & Structural Feature Detection)

This notebook extracts structural, lexical, and statistical features from email text and trains an **XGBoost Classifier**:
- **Dataset:** `phishing_email.csv` (~82,486 emails)
- **Features:** Character counts, token lengths, uppercase ratios, punctuation frequencies, security/financial keywords, URL indicators
- **Model:** Extreme Gradient Boosting (`XGBClassifier`)
- **Export:** `backend/ml/saved_models/xgb_model.json`

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve

print("Libraries successfully imported!")

## 1. Load Data and Extract Tabular Features

In [ ]:
df = pd.read_csv("phishing_email.csv")
X_text = df["text_combined"].astype(str)
y = df["label"].astype(int)

def extract_features(texts):
    feats = pd.DataFrame()
    feats["char_length"] = texts.apply(len)
    feats["word_count"] = texts.apply(lambda x: len(x.split()))
    feats["avg_word_length"] = feats["char_length"] / (feats["word_count"] + 1)
    feats["uppercase_count"] = texts.apply(lambda x: sum(1 for c in x if c.isupper()))
    feats["uppercase_ratio"] = feats["uppercase_count"] / (feats["char_length"] + 1)
    feats["digit_count"] = texts.apply(lambda x: sum(1 for c in x if c.isdigit()))
    feats["digit_ratio"] = feats["digit_count"] / (feats["char_length"] + 1)
    feats["exclamation_count"] = texts.apply(lambda x: x.count('!'))
    feats["question_count"] = texts.apply(lambda x: x.count('?'))
    feats["dollar_count"] = texts.apply(lambda x: x.count('$'))
    feats["percent_count"] = texts.apply(lambda x: x.count('%'))
    feats["url_keyword_count"] = texts.apply(lambda x: len(re.findall(r'(https?://|www\.|\.com|\.org|\.net|\.xyz|\.ru)', x, re.I)))
    feats["urgent_keyword_count"] = texts.apply(lambda x: len(re.findall(r'\b(urgent|immediately|action required|suspended|expire|overdue)\b', x, re.I)))
    feats["credential_keyword_count"] = texts.apply(lambda x: len(re.findall(r'\b(password|login|verify|account|security|ssn|bank)\b', x, re.I)))
    feats["financial_keyword_count"] = texts.apply(lambda x: len(re.findall(r'\b(invoice|payment|transfer|billing|funds|dollar|credit)\b', x, re.I)))
    return feats

print("Extracting features from text...")
features = extract_features(X_text)
features.head(5)

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")

## 3. Train XGBoost Model

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

print("Training XGBoost...")
model.fit(X_train, y_train)
print("Done training!")

## 4. Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f"XGBoost Accuracy: {acc * 100:.2f}%")
print(f"XGBoost ROC-AUC : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate (0)", "Phishing (1)"], digits=4))

## 5. Feature Importance Chart

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("XGBoost Feature Importance (Gain / Weight)")
plt.barh(range(len(indices)), importances[indices][::-1], align="center", color="teal")
plt.yticks(range(len(indices)), [features.columns[i] for i in indices][::-1])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

## 6. Save Model Artifacts

In [ ]:
save_dir = os.path.join("backend", "ml", "saved_models")
os.makedirs(save_dir, exist_ok=True)
model.save_model(os.path.join(save_dir, "xgb_model.json"))
joblib.dump(features.columns.tolist(), os.path.join(save_dir, "xgb_features.joblib"))
print("XGBoost artifacts saved successfully!")